# 🏎️ Formula 1 Exploratory Data Analysis

This notebook explores key performance trends in Formula 1 using a cleaned and feature-engineered dataset.

### Objectives

* Identify all-time race win leaders
* Analyze driver consistency using win and podium rates
* Evaluate positions gained during races
* Examine driver points trends across seasons

The insights generated in this notebook will support the development of an interactive Formula 1 Performance Analytics Dashboard.


In [ ]:
# ============================================================
# NOTEBOOK 3: Driver EDA
# ============================================================

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Load data
master = pd.read_csv("/content/drive/MyDrive/f1-analytics/data/processed/processedmaster_df.csv")
driver_stats = pd.read_csv("/content/drive/MyDrive/f1-analytics/data/processed/processeddriver_season_stats.csv")

# Filter to modern F1 era (post-1990) for relevance
# WHY? Pre-1980 races had fewer cars and different point systems.
# Modern era is more comparable and interesting to recruiters.
master_modern = master[master['year'] >= 2000].copy()
driver_stats_modern = driver_stats[driver_stats['year'] >= 2000].copy()

print(f"Modern era races: {master_modern.shape}")

In [ ]:
# ── ANALYSIS 1: ALL-TIME WIN LEADERS (2000 onwards) ──────────
#
# WHY THIS CHART: Win count is the clearest measure of success.
# This immediately shows the dominant eras (Hamilton, Schumacher).
# We use a horizontal bar chart because driver names are long text
# — horizontal bars are easier to read than rotated x-labels.

total_wins = (
    master_modern[master_modern['is_winner'] == 1]
    .groupby('driver_name')['is_winner']
    .sum()
    .sort_values(ascending=True)  # ascending=True so plotly shows highest at top
    .tail(15)                     # top 15 only, keeps chart clean
    .reset_index()
)
total_wins.columns = ['Driver', 'Wins']

fig1 = px.bar(
    total_wins,
    x='Wins',
    y='Driver',
    orientation='h',          # horizontal bars
    title='Top 15 F1 Race Winners (2000-Present)',
    color='Wins',             # color encodes win count for extra visual clarity
    color_continuous_scale='Reds',
    text='Wins'               # show win count on each bar
)

fig1.update_layout(
    plot_bgcolor='white',
    paper_bgcolor='white',
    title_font_size=18,
    xaxis_title='Number of Wins',
    yaxis_title='',
    coloraxis_showscale=False  # hide the color legend (redundant with text)
)

fig1.update_traces(textposition='outside')
fig1.show()

In [ ]:
# ── ANALYSIS 2: DRIVER CONSISTENCY — Win Rate vs Podium Rate ──
#
# WHY THIS CHART: Win count favors drivers who raced more seasons.
# Rates (%) normalize for career length. A driver with 10 wins in
# 15 races is arguably better than one with 30 wins in 200 races.
# Scatter plot reveals which drivers were efficient vs prolific.

# Focus on drivers with at least 30 race starts for statistical meaning
driver_career = driver_stats_modern.groupby('driver_name').agg(
    career_wins    = ('total_wins', 'sum'),
    career_podiums = ('total_podiums', 'sum'),
    career_races   = ('races_entered', 'sum'),
    career_points  = ('total_points', 'sum')
).reset_index()

driver_career = driver_career[driver_career['career_races'] >= 30]

driver_career['career_win_rate']    = (driver_career['career_wins'] / driver_career['career_races'] * 100).round(1)
driver_career['career_podium_rate'] = (driver_career['career_podiums'] / driver_career['career_races'] * 100).round(1)

fig2 = px.scatter(
    driver_career,
    x='career_win_rate',
    y='career_podium_rate',
    size='career_races',          # bubble size = experience
    color='career_points',        # color = total points
    hover_name='driver_name',     # show name on hover
    text='driver_name',           # label each point
    title='Driver Win Rate vs Podium Rate (Min. 30 Races, 2000-Present)',
    labels={
        'career_win_rate': 'Win Rate (%)',
        'career_podium_rate': 'Podium Rate (%)',
        'career_points': 'Career Points',
        'career_races': 'Races Entered'
    },
    color_continuous_scale='Viridis'
)

fig2.update_traces(textposition='top center', textfont_size=8)
fig2.update_layout(plot_bgcolor='white', paper_bgcolor='white', title_font_size=16)
fig2.show()

In [ ]:
# ── ANALYSIS 3: POSITIONS GAINED — OVERTAKING LEGENDS ─────────
#
# WHY THIS CHART: This is the MOST INTERESTING insight in the
# project. It separates car advantage from driver skill.
# A driver in a fast car starts on pole and wins easily.
# A driver who starts 10th and finishes 2nd is actually impressive.
# This metric is your "standout insight" in interviews.

overtakers = (
    master_modern
    .groupby('driver_name')['positions_gained']
    .agg(['mean', 'count'])
    .reset_index()
)
overtakers.columns = ['Driver', 'Avg_Positions_Gained', 'Races']
overtakers = overtakers[overtakers['Races'] >= 30]  # enough data to be meaningful
overtakers = overtakers.sort_values('Avg_Positions_Gained', ascending=False).head(20)

fig3 = px.bar(
    overtakers,
    x='Driver',
    y='Avg_Positions_Gained',
    title='Top 20 Drivers by Average Positions Gained Per Race (Min 30 Races)',
    color='Avg_Positions_Gained',
    color_continuous_scale='RdYlGn',  # Red=negative, Yellow=neutral, Green=positive
    text=overtakers['Avg_Positions_Gained'].round(2)
)

fig3.update_layout(
    xaxis_tickangle=-45,
    plot_bgcolor='white',
    paper_bgcolor='white',
    title_font_size=16,
    yaxis_title='Avg Positions Gained',
    coloraxis_showscale=False
)
fig3.update_traces(textposition='outside')
fig3.show()

In [ ]:
# ── ANALYSIS 4: DRIVER POINTS TREND OVER SEASONS ──────────────
#
# WHY THIS CHART: Line charts show trajectory. Seeing Hamilton's
# rise, Schumacher's dominance and decline, Verstappen's emergence
# tells a compelling story. This is "storytelling with data."

# Select top drivers by career points for readability
top_drivers = driver_career.nlargest(8, 'career_points')['driver_name'].tolist()

trend_data = driver_stats_modern[driver_stats_modern['driver_name'].isin(top_drivers)]

fig4 = px.line(
    trend_data,
    x='year',
    y='total_points',
    color='driver_name',
    title='Season Points Trend — Top 8 Drivers (2000-Present)',
    labels={'total_points': 'Season Points', 'year': 'Year', 'driver_name': 'Driver'},
    markers=True  # show dots at each data point
)

fig4.update_layout(
    plot_bgcolor='white',
    paper_bgcolor='white',
    title_font_size=16,
    legend_title='Driver'
)
fig4.show()

# Key Insights

**Lewis Hamilton stands out as the most successful driver** of the modern Formula 1 era with 105 race wins, significantly ahead of Max Verstappen (63 wins), Michael Schumacher (56 wins), and Sebastian Vettel (53 wins). This highlights Hamilton's exceptional longevity and dominance across multiple seasons.

The positions gained analysis reveals a different dimension of driver performance. Drivers such as **Tiago Monteiro**, **Jos Verstappen**, and **Mika Salo** gained the most positions on average relative to their starting grid *italicized text* positions, demonstrating strong racecraft, overtaking ability, and adaptability during races.